# HeartShare Table One — BDC runtime notebook

This thin notebook reads the runtime archive recorded by the harmonization
run, verifies it, and imports the shared Table One implementation. Set
`RUNTIME_BUNDLE_PATH` and `RUNTIME_BUNDLE_SHA256` when an older run needs an
updated Table One runtime, or use the updated self-contained
`heartshare_table_one.ipynb` with the same run folder. Step 3 supports cohort
filters, subgroup comparisons, and per-study longitudinal tables and plots.


## 1. Where is the run?


In [ ]:
# ---------------------------------------------------------------------------
# 1. WHERE IS THE RUN? — the output folder of a harmonization run.
# ---------------------------------------------------------------------------
from pathlib import Path

HARMONIZATION_OUTPUT_ROOT = Path("/sbgenomics/workspace/output-files")

# Copy the exact output folder printed by Step 5 of the harmonization notebook.
# Do not leave this as None: Table One reads that run's manifest and long data.
RUN_DIR = None
# RUN_DIR = "/sbgenomics/workspace/output-files/harmonized_20260812T174618Z"

_available_runs = sorted(
    (
        _path
        for _path in HARMONIZATION_OUTPUT_ROOT.glob("harmonized_*")
        if (_path / "manifest.json").is_file()
    ),
    key=lambda _path: _path.stat().st_mtime,
    reverse=True,
)
print("Available harmonization runs (newest first):")
if _available_runs:
    for _path in _available_runs[:20]:
        print(f"  {_path}")
else:
    print(f"  none found under {HARMONIZATION_OUTPUT_ROOT}")
if RUN_DIR is None:
    print("\nSet RUN_DIR above to one of these folders, then rerun from Step 1.")

# Where the table and figures go. None puts them in a table_one/ folder inside
# the run, which keeps a run and its summary together.
OUTPUT_DIR = None

# Optional verified runtime override for an older run or newer Table One features.
RUNTIME_BUNDLE_PATH = None
RUNTIME_BUNDLE_SHA256 = None


In [ ]:
"""Small standard-library bootstrap used by HeartShare thin BDC notebooks."""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def runtime_spec_from_release(manifest_path: Path) -> dict[str, object]:
    try:
        import yaml
    except ImportError as exc:
        raise RuntimeError("PyYAML is required to read the release manifest") from exc
    document = yaml.safe_load(Path(manifest_path).read_text(encoding="utf-8")) or {}
    retired = document.get("retired")
    if retired and os.environ.get("HEARTSHARE_ALLOW_RETIRED_RELEASE") != "1":
        raise RuntimeError(
            f"Release {document.get('release_version')} is RETIRED: {retired}. Use the release that "
            "supersedes it (set HEARTSHARE_ALLOW_RETIRED_RELEASE=1 only to reproduce an old result)."
        )
    spec = document.get("runtime_bundle")
    if not isinstance(spec, dict) or not spec.get("path") or not spec.get("sha256"):
        raise ValueError(
            f"Release manifest has no complete runtime_bundle path/checksum: {manifest_path}"
        )
    resolved = dict(spec)
    path = Path(str(spec["path"]))
    if not path.is_absolute():
        path = (Path(manifest_path).parent / path).resolve()
    resolved["path"] = path
    # What this release's mapping payload needs from its runtime (written by the release manager).
    resolved["required_runtime_features"] = list(document.get("required_runtime_features") or [])
    return resolved


def _safe_extract(archive_path: Path, target: Path) -> None:
    with ZipFile(archive_path) as archive:
        root = target.resolve()
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f"Unsafe path in runtime archive: {member.filename}")
        archive.extractall(target)


def _verify_extracted(target: Path) -> dict[str, object]:
    manifest_path = target / "runtime_manifest.json"
    if not manifest_path.exists():
        raise ValueError("Runtime archive is missing runtime_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in (manifest.get("files") or {}).items():
        path = target / name
        if not path.is_file():
            raise ValueError(f"Runtime archive is missing declared file: {name}")
        actual = sha256_file(path)
        if actual != expected:
            raise ValueError(
                f"Runtime file checksum mismatch for {name}: expected {expected}, got {actual}"
            )
    return manifest


def _install_offline_wheels(target: Path, manifest: dict[str, object]) -> Path | None:
    requirements = list(manifest.get("required_offline_distributions") or [])
    if not requirements:
        return None
    wheelhouse = target / "wheelhouse"
    vendor = target / "vendor"
    marker = vendor / ".heartshare_wheels_installed"
    if marker.exists():
        return vendor
    vendor.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-index",
        "--find-links",
        str(wheelhouse),
        "--target",
        str(vendor),
        *requirements,
    ]
    subprocess.run(command, check=True)
    marker.write_text("ok\n", encoding="utf-8")
    return vendor


def bootstrap_runtime(
    archive_path: Path,
    expected_sha256: str,
    *,
    workspace_root: Path | None = None,
    expected_output_schema: str = "1.1",
    required_runtime_features: list[str] | tuple[str, ...] = (),
    supported_output_schemas: tuple[str, ...] | None = None,
) -> tuple[Path, dict[str, object]]:
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Runtime bundle not found: {archive_path}")
    actual_sha256 = sha256_file(archive_path)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Runtime bundle checksum mismatch: "
            f"expected {expected_sha256}, got {actual_sha256}"
        )

    workspace_root = Path(
        workspace_root
        or os.environ.get(
            "HEARTSHARE_VENDOR_ROOT", "/sbgenomics/workspace/_heartshare_vendor"
        )
    )
    workspace_root.mkdir(parents=True, exist_ok=True)
    target = workspace_root / actual_sha256[:16]
    if not (target / "runtime_manifest.json").exists():
        with tempfile.TemporaryDirectory(
            prefix="heartshare-runtime-", dir=workspace_root
        ) as temporary:
            staging = Path(temporary) / "unpacked"
            staging.mkdir()
            _safe_extract(archive_path, staging)
            _verify_extracted(staging)
            try:
                staging.rename(target)
            except FileExistsError:
                shutil.rmtree(staging)

    manifest = _verify_extracted(target)
    supported = supported_output_schemas or (expected_output_schema,)
    if str(manifest.get("output_schema_version")) not in supported:
        raise ValueError(
            "Runtime/output schema mismatch: "
            f"expected one of {supported}, runtime provides "
            f"{manifest.get('output_schema_version')}"
        )

    unsupported = sorted(set(required_runtime_features) - set(manifest.get("runtime_features") or ()))
    if unsupported:
        raise ValueError(
            "This release's mappings need runtime features the bundled runtime "
            f"({manifest.get('runtime_version')}) does not provide: {unsupported}. The release is "
            "mis-assembled — do not run it; rebuild it with a current runtime bundle."
        )

    vendor = _install_offline_wheels(target, manifest)
    import_root = target / str(manifest.get("import_root") or "bdc/src")
    for path in [vendor, target, import_root]:
        if path is not None and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    os.environ["HEARTSHARE_RUNTIME_BUNDLE_PATH"] = str(archive_path)
    os.environ["HEARTSHARE_RUNTIME_BUNDLE_SHA256"] = actual_sha256
    os.environ["HEARTSHARE_RUNTIME_VERSION"] = str(
        manifest.get("runtime_version") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_SHA"] = str(
        manifest.get("build_git_sha") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_BRANCH"] = str(
        manifest.get("build_git_branch") or "runtime-bundle"
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_CLEAN"] = str(
        bool(manifest.get("build_git_clean", True))
    ).lower()
    return import_root, manifest


if not RUN_DIR:
    raise ValueError(
        "RUN_DIR is not set. In Step 1, choose one of the listed harmonization "
        "run folders, set RUN_DIR to its full path, and rerun from Step 1."
    )
_run_manifest_path = Path(RUN_DIR) / "manifest.json"
if not _run_manifest_path.is_file():
    raise FileNotFoundError(
        f"No harmonization manifest found at {_run_manifest_path}. RUN_DIR must "
        "be the exact output folder created by Step 5 of the harmonization "
        "notebook, not the output-files parent folder or the example path."
    )
_run_manifest = json.loads(_run_manifest_path.read_text(encoding="utf-8"))
_runtime_spec = dict(_run_manifest.get("runtime_bundle") or {})
_runtime_path = RUNTIME_BUNDLE_PATH or _runtime_spec.get("path")
_runtime_sha256 = RUNTIME_BUNDLE_SHA256 or _runtime_spec.get("sha256")
if not _runtime_path or not _runtime_sha256:
    raise ValueError(
        "This run does not record a runtime bundle. Set RUNTIME_BUNDLE_PATH and "
        "RUNTIME_BUNDLE_SHA256 in Step 1, or use heartshare_table_one.ipynb."
    )
_runtime_root, _runtime_manifest = bootstrap_runtime(
    Path(_runtime_path), str(_runtime_sha256),
    supported_output_schemas=("1.1", "1.2"),
    required_runtime_features=("table_one/analysis/1",)
)
import table_one as t1
print(f"Runtime {_runtime_manifest['runtime_version']} ready")


In [ ]:
# Loads the run. Nothing to edit here.
if not RUN_DIR:
    raise ValueError(
        "RUN_DIR is not set. In Step 1, choose one of the listed harmonization "
        "run folders, set RUN_DIR to its full path, and rerun from Step 1."
    )

run_dir = Path(RUN_DIR)
if not run_dir.is_dir():
    raise FileNotFoundError(
        f"RUN_DIR does not exist: {run_dir}. Choose a folder listed in Step 1."
    )
output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else run_dir / "table_one"

files = t1.find_run_files(run_dir)
long_df = t1.load_long(files["long"])
variable_dictionary = t1.load_variable_dictionary(files["variable_dictionary"])

print(f"Run    : {run_dir}")
print(f"Data   : {files['long'].name}  ({len(long_df):,} rows)")
print(f"Meta   : {'variable_dictionary.csv' if variable_dictionary is not None else 'NOT FOUND — variable types will be inferred'}")
print(f"Output : {output_dir}")


## 2. What is in it?


In [ ]:
# ---------------------------------------------------------------------------
# 2. WHAT IS IN IT? — read-only. Lists what you can summarize.
# ---------------------------------------------------------------------------
_all_meta = t1.order_variables(
    t1.build_variable_metadata(long_df, variable_dictionary)
)

print(f"STUDIES ({long_df['study'].nunique()})")
print("-" * 62)
for _study, _block in long_df.groupby("study"):
    print(f"  {_study:<22} {_block['participant_id'].nunique():>6,} participants"
          f"   {_block['standard_name'].nunique():>3} variables")

_timepoint_column = (
    "timepoint" if "timepoint" in long_df.columns and long_df["timepoint"].notna().any()
    else "exam_label" if "exam_label" in long_df.columns else None
)
if _timepoint_column:
    _timepoint_counts = (
        long_df.groupby(_timepoint_column, dropna=False)["participant_id"]
        .nunique()
        .sort_index()
    )
    print(f"\nTIMEPOINTS ({len(_timepoint_counts)})")
    print("-" * 62)
    for _timepoint, _count in _timepoint_counts.items():
        print(f"  {str(_timepoint):<30} {_count:>6,} participants")
    if len(_timepoint_counts) > 1:
        print("\nPARTICIPANTS BY STUDY AND TIMEPOINT")
        print("-" * 62)
        _study_timepoints = (
            long_df.groupby(["study", _timepoint_column], dropna=False)
            ["participant_id"].nunique()
        )
        for (_study, _timepoint), _count in _study_timepoints.items():
            print(f"  {_study:<20} {str(_timepoint):<22} {_count:>6,}")

print(f"\nVARIABLES ({len(_all_meta)})")
print("-" * 62)
_current_domain = None
for _meta in _all_meta:
    if _meta.domain != _current_domain:
        _current_domain = _meta.domain
        print(f"\n  [{_current_domain or 'other'}]")
    _flag = "" if _meta.kind_source == "variable_dictionary" else "  (type inferred)"
    print(f"    {_meta.standard_name:<24} {_meta.kind:<12}{_flag}")


## 3. Choose what to summarize


In [ ]:
# ---------------------------------------------------------------------------
# 3. CHOOSE — variables, studies, and totals are optional.
# ---------------------------------------------------------------------------

# Single-visit table timepoint, or eligibility/subgroup visit in longitudinal mode.
# Set None only when the run contains exactly one timepoint, or choose another
# listed value such as "week_08" for a follow-up-specific table.
TIMEPOINT = "baseline"

# Which variables, in the order you want them. Empty = every variable found.
VARIABLES = [
    # "age",
    # "sex",
    # "bmi",
]

# Which studies become columns. Empty = every study in the run.
STUDIES = []

# Optional participant subset, evaluated at TIMEPOINT before VARIABLES selection.
# None = everyone. Keep all measurements for participants who meet the condition.
COHORT_FILTER = None
# COHORT_FILTER = ("bmi", ">=", 30)       # BMI >= 30 kg/m²
# COHORT_FILTER = ("bmi", "<", 30)        # complementary BMI group
# COHORT_FILTER = ("afib", "in", ["Yes", "1", "1.0"])  # AFib only
# Check your run's AFib labels/codes: these examples assume 1 = Yes.
# Missing trait values are excluded, not assigned to the comparison group.
# Multiple conditions are combined with AND (including COHORT_FILTER if set).
COHORT_FILTERS = []
# COHORT_FILTERS = [("bmi", ">=", 30), ("afib", "in", ["Yes", "1", "1.0"])]

# Side-by-side subgroup columns, separately for each study.
STRATIFY = None
# STRATIFY = {"trait": "bmi", "cutoff": 30}
# STRATIFY = {"trait": "afib", "groups": {
#     "No AFib": ["No", "0", "0.0"], "AFib": ["Yes", "1", "1.0"]}}
# Missing/unlisted categories are excluded; subgroup definitions must not overlap.

# Separate tables and plots for each study, showing that study's visits.
LONGITUDINAL = False
# LONGITUDINAL = True
# Empty = all available visits per study, baseline first then duration/natural order.
# Supply lists to select/order visits, especially for labels such as visit_1.
TIMEPOINTS_BY_STUDY = {}
# TIMEPOINTS_BY_STUDY = {"STUDY-A": ["baseline", "week_08", "week_16"]}
# With STRATIFY, longitudinal output has one panel per study AND subgroup.
# Eligibility/group membership stays fixed at TIMEPOINT; each visit uses available cases.

# Default median [IQR]; choose mean ± sample SD globally or for individual variables.
CONTINUOUS_SUMMARIES = {}
# CONTINUOUS_SUMMARIES = {"*": "median_iqr", "age": "mean_sd"}
# Use different OUTPUT_DIR folders when saving different analysis configurations.

# Known enrolled N per study. Partial dictionaries are allowed: omitted studies
# use observed lower-bound counts and are labelled that way in every export.
STUDY_TOTALS = {
    # "HFN-NEAT": 100,
    # "TOPCAT": 3445,
}

# Add a pooled column across all studies.
INCLUDE_OVERALL = True

# Which files to write in step 6. Delete the ones you don't want.
EXPORT_FORMATS = [
    "csv",
    "xlsx",
    "docx",
    "html",
]


## 4. Table 1


In [ ]:
# ---------------------------------------------------------------------------
# 4. TABLES AND PARTICIPANT SELECTION
# ---------------------------------------------------------------------------
try:
    from IPython.display import display
except ImportError:
    display = print

if not hasattr(t1, "build_analysis"):
    raise RuntimeError(
        "This run's runtime predates subgroup/longitudinal Table One support. "
        "Use the updated self-contained heartshare_table_one.ipynb for this run, "
        "or set RUNTIME_BUNDLE_PATH and RUNTIME_BUNDLE_SHA256 to an updated runtime."
    )
try:
    analysis = t1.build_analysis(
        long_df, variable_dictionary, timepoint=TIMEPOINT, studies=STUDIES,
        variables=VARIABLES, cohort_filter=COHORT_FILTER, cohort_filters=COHORT_FILTERS,
        stratify=STRATIFY, longitudinal=LONGITUDINAL,
        timepoints_by_study=TIMEPOINTS_BY_STUDY, study_totals=STUDY_TOTALS,
        include_overall=INCLUDE_OVERALL, continuous_summaries=CONTINUOUS_SUMMARIES,
    )
except t1.DuplicateParticipantValuesError as _error:
    display(_error.report)
    raise

analysis_df = analysis.analysis_df
print("PARTICIPANT SELECTION (counts from harmonized data)")
display(analysis.selection)
for _panel in analysis.panels:
    print(_panel.title)
    display(_panel.result.table)
    for _warning in _panel.result.warnings:
        print(f"  - {_warning}")
    if not _panel.result.duplicate_report.empty:
        display(_panel.result.duplicate_report)

# Convenient handles for inspecting the first panel interactively.
result = analysis.panels[0].result
participant_values = analysis.panels[0].values


## 5. Figures


In [ ]:
# ---------------------------------------------------------------------------
# 5. FIGURES — each panel uses the same participants/visits as its table.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

saved = []
for _panel in analysis.panels:
    _panel_dir = output_dir if _panel.key == "table_one" else output_dir / _panel.key
    figure_dir = _panel_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    for _meta in _panel.result.variables:
        if _meta.kind == "continuous":
            _figure = t1.plot_continuous(_panel.values, _meta, _panel.result.groups)
            _name = f"box_{t1.artifact_key(_meta)}.png"
        else:
            _figure = t1.plot_categorical(_panel.values, _meta, _panel.result.groups)
            _name = f"bar_{t1.artifact_key(_meta)}.png"
        _figure.axes[0].set_xlabel(_panel.axis_label)
        _figure.suptitle(_panel.title, fontsize=10)
        _figure.tight_layout()
        _figure.savefig(figure_dir / _name, bbox_inches="tight", facecolor="white")
        saved.append(str(figure_dir / _name))
        plt.show()
        plt.close(_figure)

        if LONGITUDINAL and _meta.kind == "continuous":
            _figure = t1.plot_visit_summary(_panel, _meta)
            _path = figure_dir / f"trend_{t1.artifact_key(_meta)}.png"
            _figure.savefig(_path, bbox_inches="tight", facecolor="white")
            saved.append(str(_path))
            plt.show()
            plt.close(_figure)

    for _kind, _name in (("continuous", "overview_continuous.png"),
                         ("categorical", "overview_categorical.png")):
        _figure = t1.plot_overview(_panel.values, _panel.result.variables, _panel.result.groups, _kind)
        if _figure is None:
            continue
        _figure.suptitle(_panel.title, fontsize=10)
        _figure.tight_layout()
        _figure.savefig(figure_dir / _name, bbox_inches="tight", facecolor="white")
        saved.append(str(figure_dir / _name))
        plt.show()
        plt.close(_figure)

print(f"Wrote {len(saved)} figure(s) under {output_dir}")


## 6. Export


In [ ]:
# ---------------------------------------------------------------------------
# 6. EXPORT — tables, analysis settings, and participant selection counts.
# ---------------------------------------------------------------------------
written = {}
for _panel in analysis.panels:
    _panel_dir = output_dir if _panel.key == "table_one" else output_dir / _panel.key
    _paths = t1.export_table(
        _panel.result, _panel_dir, formats=EXPORT_FORMATS,
        title=f"Table 1. {_panel.title}",
    )
    for _format, _path in _paths.items():
        _key = _format if _panel.key == "table_one" else f"{_panel.key}/{_format}"
        written[_key] = _path
        print(_path)

_sources = dict(files)
_sources["manifest"] = run_dir / "manifest.json"
t1.save_analysis_metadata(analysis, output_dir, _sources, EXPORT_FORMATS)
print(f"Settings: {output_dir / 'analysis_settings.json'}")
print(f"Selection counts: {output_dir / 'participant_selection.csv'}")
